# Differentially Private Covariance Estimation

Reproducing Section 4 experiments: **Algorithm 1** vs **Laplace / Gaussian / KT** baselines on Wine and Adult datasets.

**Key conventions (from paper)**
- Data matrix `X ∈ R^{d×n}`: columns are samples, rows are features.
- Target: estimate `C = XX^T` (unnormalized covariance, Sec 2).
- All mechanisms output normalized `Ĉ = C̃/n` with eigenvalues clipped to `[0,1]`.
- Error metric: normalized Frobenius distance `‖Ĉ − C‖_F / ‖C‖_F` (Sec 4).

## 1. Imports

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eigh
from sklearn.datasets import fetch_openml, load_wine

warnings.filterwarnings("ignore")

## 2. Data Loading and Preprocessing

In [ ]:
def _show_progress(step, total, label):
    """Print a simple one-line progress bar for long dataset preparation steps."""
    width = 28
    filled = int(width * step / total)
    bar = "#" * filled + "-" * (width - filled)
    pct = int(100 * step / total)
    print(f"\r[{bar}] {pct:3d}% {label}", end="", flush=True)
    if step == total:
        print()


def _standardize_and_normalize(df):
    """Standard ML preprocessing → paper's d×n layout with unit-norm columns."""
    X = df.values.astype(float)
    X = X[:, X.std(axis=0) > 0]                        # drop zero-variance features
    X = (X - X.mean(axis=0)) / X.std(axis=0)           # standardize features
    X = X.T                                             # n×d → d×n  (paper convention)
    col_norms = np.linalg.norm(X, axis=0, keepdims=True)
    X = X / np.maximum(col_norms, 1.0)                 # clip column norms to ≤ 1
    return X


def load_wine_appendix():
    """Wine dataset: sklearn's 13-feature wine data."""
    X, _ = load_wine(return_X_y=True, as_frame=True)
    return _standardize_and_normalize(X)


def load_airfoil_appendix():
    """Airfoil data from local CSV."""
    df = pd.read_csv("data/airfoil.csv", header=None)
    return _standardize_and_normalize(df)


def load_adult_appendix():
    """
    Adult dataset (OpenML), one-hot encoded.
    Yields ~105 features — closest to the paper's 108-dim setup.
    """
    total_steps = 5
    cache_dir = "data/.sk_cache"
    _show_progress(0, total_steps, "Adult: fetching OpenML dataset")
    adult = fetch_openml("adult", version=2, as_frame=True, data_home=cache_dir)
    _show_progress(1, total_steps, "Adult: copying raw frame")
    X = adult.data.copy()
    _show_progress(2, total_steps, "Adult: one-hot encoding categorical columns")
    X = pd.get_dummies(X, drop_first=False)
    _show_progress(3, total_steps, "Adult: standardizing features")
    X = X.values.astype(float)
    X = X[:, X.std(axis=0) > 0]
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    _show_progress(4, total_steps, "Adult: transposing and clipping samples")
    X = X.T
    col_norms = np.linalg.norm(X, axis=0, keepdims=True)
    X = X / np.maximum(col_norms, 1.0)
    _show_progress(5, total_steps, f"Adult: ready with shape {X.shape}")
    return X


def subsample(X, n, rng):
    idx = rng.choice(X.shape[1], size=min(n, X.shape[1]), replace=False)
    return X[:, idx]

In [ ]:
# Load datasets
X_wine = load_wine_appendix()
print(f"Wine:    {X_wine.shape}")

X_adult = load_adult_appendix()
print(f"Adult:   {X_adult.shape}")

## 3. Covariance Matrices and Evaluation Metric

In [ ]:
def true_covariance(X):
    """Normalized covariance C = XX^T / n used as ground truth (Sec 2)."""
    # Each column of X has l2-norm ≤ 1 (paper convention)
    return X @ X.T / X.shape[1]


def gram_matrix(X):
    """Unnormalized covariance C = XX^T ∈ R^{d×d} — input to all DP mechanisms (Sec 2)."""
    return X @ X.T


def frobenius_error(C_true, C_hat):
    # Paper (Sec 4): normalized Frobenius distance ‖Ĉ − C‖_F / ‖C‖_F
    return np.linalg.norm(C_true - C_hat, "fro") / np.linalg.norm(C_true, "fro")


def _clip_psd_full(C_hat):
    """
    Project onto PSD cone: symmetrize then clip eigenvalues to [0, 1].

    Paper (Sec 4): eigenvalue rounding applied to all algorithm outputs;
    clip to [0, n] before /n normalization, i.e. [0, 1] afterwards.
    """
    C_hat = (C_hat + C_hat.T) / 2.0      # symmetrize (absorb float asymmetry)
    eigvals, eigvecs = eigh(C_hat)
    eigvals = np.clip(eigvals, 0.0, 1.0) # clip to [0, 1]  (paper: [0, n] before /n)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T


def _clip_normalized_eigenvalues(C_hat):
    """Same as _clip_psd_full without the symmetrization step."""
    eigvals, eigvecs = eigh(C_hat)
    eigvals = np.clip(eigvals, 0.0, 1.0)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T

## 4. Exponential Mechanism (shared sampler)

Paper (Sec 2, Lemma 4 / Alg 1 Step 2a): sample $\hat{u}$ from density
$$f_{C_i}(u) \propto \exp\!\left(\frac{\varepsilon_i}{4}\, u^\top C_i\, u\right) \quad u \in S^{d-1}$$
The denominator 4 comes from sensitivity $\Delta_g = 2$ (unit-norm columns).

The exact sampler is **Algorithm 2** (Kent et al. rejection sampler, Sec 3.2).  
Here we use a Monte-Carlo softmax approximation over random unit vectors.

In [ ]:
def _sample_exponential_mechanism(C, epsilon, rng, num_candidates=500):
    """
    Approximate exponential mechanism on S^{d-1}.

    Exact density (Alg 1 Step 2a):  f(u) ∝ exp( (ε/4) * u^T C u )
    Approximation: softmax over `num_candidates` uniform random unit vectors.
    """
    d = C.shape[0]
    # Sample uniform unit vectors on S^{d-1}
    candidates = rng.standard_normal((num_candidates, d))
    candidates /= np.linalg.norm(candidates, axis=1, keepdims=True)
    # Utility score g(C, u) = u^T C u  (Alg 1 Step 2a)
    scores = np.einsum("ij,jk,ik->i", candidates, C, candidates)
    # Weights ∝ exp( (ε_i/4) * u^T C u )  (Lemma 4 with Δ_g = 2)
    logits = (epsilon / 4.0) * scores
    logits -= np.max(logits)  # numerical stability
    weights = np.exp(logits)
    weights /= np.sum(weights)
    return candidates[rng.choice(num_candidates, p=weights)]

## 5. Baseline: Laplace Mechanism (Sec 4, 'L')

Add i.i.d. $\mathrm{Lap}(2d/\varepsilon)$ noise to each entry of $C = XX^\top$.

Scale $2d/\varepsilon$: the $\ell_1$-sensitivity of $C$ is $2d$ for unit-norm columns (Sec 2).

In [ ]:
def dp_laplace(X, epsilon, k, rng):
    del k  # Laplace acts on the full matrix; rank truncation not used
    d, n = X.shape
    C = gram_matrix(X)                                       # C = XX^T  (Sec 2)
    noise = rng.laplace(0.0, (2.0 * d) / epsilon, C.shape)  # Lap(2d/ε) per entry
    noise = (noise + noise.T) / 2                            # symmetrize
    return _clip_psd_full((C + noise) / n)                   # normalize + PSD clip

## 6. Baseline: Gaussian Mechanism (Sec 3.1, 'G')

$(\varepsilon, \delta)$-DP. Error bound (Sec 3.1):
$$\|C - \hat{C}\|_F \le O\!\left(\frac{d^{3/2}\sqrt{\log(1/\delta)}}{\varepsilon}\right)$$
Evaluated at $\delta \in \{10^{-16}, 10^{-10}, 10^{-3}\}$ (Sec 4).

In [ ]:
def dp_gaussian(X, epsilon, delta, k, rng):
    del k  # Gaussian acts on the full matrix; rank truncation not used
    _, n = X.shape
    C = gram_matrix(X)                                        # C = XX^T  (Sec 2)
    d = C.shape[0]
    sigma = (d ** 1.5 * np.sqrt(np.log(1.0 / delta))) / epsilon  # σ = d^{3/2}√log(1/δ)/ε
    noise = rng.standard_normal(C.shape)
    noise = (noise + noise.T) / 2                             # symmetrize
    return _clip_psd_full((C + sigma * noise) / n)            # normalize + PSD clip

## 7. Baseline: Kapralov-Talwar (Sec 4, 'KT')

Pure $\varepsilon$-DP. Iterates $k$ rounds of **rank-one subtraction** (vs. Algorithm 1's orthogonal projection).  
Per-round budget split (Lemma 2 composition):
$$\varepsilon_{\text{per}} = \varepsilon/k, \quad \varepsilon_{ev} = \varepsilon_{lam} = \varepsilon_{\text{per}}/2$$

In [ ]:
def dp_kt(X, epsilon, delta, k, rng):
    del delta  # pure ε-DP; δ not used
    if rng is None:
        rng = np.random.default_rng()

    d, n = X.shape
    if k is None:
        k = d

    C_s = gram_matrix(X).copy()  # residual matrix, starts as C = XX^T
    C_hat = np.zeros((d, d))

    # Budget split: ε_per = ε/k  (Lemma 2 composition over k rounds)
    eps_per = epsilon / k
    eps_ev  = eps_per / 2.0  # eigenvector budget (exponential mechanism)
    eps_lam = eps_per / 2.0  # eigenvalue budget  (Laplace mechanism)

    for _ in range(k):
        # Sample top eigenvector of residual: density ∝ exp( (ε_ev/4) * u^T C_s u )
        g = _sample_exponential_mechanism(C_s, eps_ev, rng)

        # Noisy eigenvalue: λ̂ = g^T C_s g + Lap(1/ε_lam)  (sensitivity = 1)
        lam = float(g @ C_s @ g) + rng.laplace(0.0, 1.0 / eps_lam)
        lam = max(lam, 0.0)                    # eigenvalue rounding (Sec 4)

        C_hat += lam * np.outer(g, g)          # accumulate rank-1 term
        C_s   = C_s - lam * np.outer(g, g)    # rank-one subtraction (key KT step)
        C_s   = (C_s + C_s.T) / 2.0           # re-symmetrize residual

    # Ĉ = (1/n) Σ_i λ̂_i g_i g_i^T,  then PSD clip  (Sec 4)
    return _clip_psd_full(C_hat / n)

## 8. Algorithm 1: Iterative Eigenvector Sampling (Sec 3)

**Full pseudocode (paper Sec 3):**

| Step | Operation |
|------|-----------|
| Init | $C_1 = C$, $P_1 = I_d$ |
| Step 1 | $\hat{\lambda}_i = \lambda_i(C) + \mathrm{Lap}(2/\varepsilon_0)$, $\;\varepsilon_0 = \varepsilon/2$ |
| Step 2a | $\hat{u}_i \sim \exp\!\left(\frac{\varepsilon_i}{4} u^\top C_i u\right)$ on $S^{d-i}$, $\;\hat{\theta}_i = P_i^\top \hat{u}_i$ |
| Step 2b | $P_{i+1}$: orthonormal basis of $\{\hat{\theta}_1,\ldots,\hat{\theta}_i\}^\perp$ |
| Step 2c | $C_{i+1} = P_{i+1}\, C\, P_{i+1}^\top$ |
| Step 3 | $\hat{C} = \sum_i \hat{\lambda}_i \hat{\theta}_i \hat{\theta}_i^\top / n$ |

**Privacy** (Theorem 1): $(\sum_{i=0}^d \varepsilon_i)$-DP by composition (Lemma 2).  
**Utility** (Theorem 2): $\|C-\hat{C}\|_F \le \tilde{O}\!\left(\sqrt{\sum_i d\lambda_i/\varepsilon_i + \sqrt{d}/\varepsilon_0}\right)$

In [ ]:
def _orth_complement(g):
    """
    Return orthonormal basis of g⊥  (Alg 1 Step 2b).

    SVD of g (column vector): U[:,0] = g,  U[:,1:] spans g⊥.
    """
    U, _, _ = np.linalg.svd(g.reshape(-1, 1), full_matrices=True)
    return U[:, 1:]  # shape (d-i) × (d-i-1): orthonormal basis of û_i⊥


def _algorithm1_privacy_split(C, epsilon, rng, beta=0.01, adaptive=False):
    """
    Compute noisy eigenvalues and per-round budgets.

    Step 1 (both variants):  ε_0 = ε/2,  λ̂_i = λ_i(C) + Lap(2/ε_0)
      → Laplace scale 2/ε_0 from l1-sensitivity ≤ 2 (Theorem 1 proof).

    Adaptive (Corollary 1, 'AD'):
      τ = (2/ε_0) log(2d/β),  ε_i = (ε/2) · √(λ̂_i+τ) / Σ_j √(λ̂_j+τ)

    Uniform (Supplement IT-U):
      ε_i = ε/(2d)  for all i
    """
    d = C.shape[0]
    eps0 = epsilon / 2.0                    # ε_0 = ε/2  (Corollary 1)

    eigvals, _ = eigh(C)
    eigvals = eigvals[::-1]                 # descending: λ_1 ≥ ... ≥ λ_d
    # Alg 1 Step 1: λ̂_i = λ_i(C) + Lap(2/ε_0)
    noisy_eigvals = eigvals + rng.laplace(0.0, 2.0 / eps0, size=d)

    if adaptive:
        # Corollary 1: τ = (4/ε) log(2d/β)
        tau = (2.0 / eps0) * np.log(2.0 * d / beta)
        weights = np.sqrt(np.maximum(noisy_eigvals + tau, 1e-12))
        # ε_i = (ε/2) · w_i / Σ_j w_j,  w_i = √(λ̂_i + τ)
        eps_rounds = (epsilon / 2.0) * weights / np.sum(weights)
    else:
        # IT-U: ε_i = ε/(2d) uniformly  (Supplement)
        eps_rounds = np.full(d, epsilon / (2.0 * d))

    return noisy_eigvals, eps_rounds


def _dp_algorithm1_core(X, epsilon, rng=None, beta=0.01, adaptive=False, rank_k=None):
    """
    Core of Algorithm 1 (Sec 3).

    Step 1 → _algorithm1_privacy_split
    Step 2a → _sample_exponential_mechanism
    Step 2b → _orth_complement
    Step 2c → project C into new subspace
    Step 3  → reconstruct Ĉ = Σ_i λ̂_i θ̂_i θ̂_i^T / n
    """
    if rng is None:
        rng = np.random.default_rng()

    d, n = X.shape
    C = gram_matrix(X)                # C = XX^T  (Sec 2)
    noisy_eigvals, eps_rounds = _algorithm1_privacy_split(
        C, epsilon, rng, beta=beta, adaptive=adaptive
    )
    # Rank-k truncation (Theorem 2 rank-k variant): stop loop at round k
    n_rounds      = d if rank_k is None else min(rank_k, d)
    noisy_eigvals = noisy_eigvals[:n_rounds]
    eps_rounds    = eps_rounds[:n_rounds]

    # Alg 1 Step 1 init: P_1 = I_d,  C_1 = C
    P_i = np.eye(d)  # P_i ∈ R^{d×(d-i+1)}: orthonormal basis of current subspace
    C_i = C.copy()   # C_i = P_i C P_i^T ∈ R^{(d-i+1)×(d-i+1)}
    thetas = []

    for i in range(n_rounds):
        # Step 2a: û_i ~ exp. mechanism on S^{d-i}
        u_i = _sample_exponential_mechanism(C_i, eps_rounds[i], rng)
        # θ̂_i = P_i^T û_i  (lift from subspace coords to R^d)
        theta_i = P_i @ u_i
        thetas.append(theta_i)

        if C_i.shape[0] == 1:
            break  # 1-dim subspace: nothing left

        # Step 2b: P_{i+1} = P_i · orth_complement(û_i)
        P_i = P_i @ _orth_complement(u_i)
        # Step 2c: C_{i+1} = P_{i+1} C P_{i+1}^T
        C_i = P_i.T @ C @ P_i

    # Step 3: Ĉ = Σ_i λ̂_i θ̂_i θ̂_i^T  (Alg 1 Step 3)
    C_hat = np.zeros((d, d))
    for lam, theta in zip(noisy_eigvals, thetas):
        C_hat += lam * np.outer(theta, theta)
    # Normalize by n and clip to PSD  (Sec 4: eigenvalue rounding)
    return _clip_psd_full(C_hat / n)


def dp_algorithm1_uniform(X, epsilon, delta, k, rng=None, beta=0.01):
    """IT-U variant: uniform budget split ε_i = ε/(2d)  (Supplement)."""
    del delta, k
    return _dp_algorithm1_core(X, epsilon, rng=rng, beta=beta, adaptive=False)


def dp_algorithm1_adaptive(X, epsilon, delta, k, rng=None, beta=0.01):
    """AD variant: adaptive budget split per Corollary 1."""
    del delta
    return _dp_algorithm1_core(X, epsilon, rng=rng, beta=beta, adaptive=True, rank_k=k)


def dp_algorithm1_rank_k(X, epsilon, delta, k, rng=None):
    """Main experiment entry point: AD variant (used in Sec 4)."""
    return dp_algorithm1_adaptive(X, epsilon, delta, k, rng=rng)

## 9. Experiment Runner

Reproduces the supplement-style two-panel comparison (Sec 4):
- **Panel (a)**: Laplace / KT / AD across ε values
- **Panel (b)**: Alg 1 vs Gaussian at several δ values across ε

In [ ]:
def run_experiment_vary_eps_gaussian_grid(
    X_full, epsilons, n, gaussian_deltas, k, n_trials=5, seed=42
):
    """
    Returns results_a (Laplace/KT/Alg1), results_b (G-δ/Alg1),
    and timing (total wall-clock seconds per algorithm).
    """
    results_a = {alg: [] for alg in ["Laplace", "KT", "Alg1"]}
    results_b = {f"G-{int(np.log10(delta))}": [] for delta in gaussian_deltas}
    results_b["Alg1"] = []

    time_totals = {alg: 0.0 for alg in ["Laplace", "Gaussian", "KT", "Alg1"]}
    time_counts = {alg: 0 for alg in time_totals}

    total_runs = len(epsilons) * n_trials
    completed_runs = 0

    for eps_idx, epsilon in enumerate(epsilons, start=1):
        errs_a = {alg: [] for alg in results_a}
        errs_b = {alg: [] for alg in results_b}

        for trial in range(n_trials):
            _show_progress(
                completed_runs, total_runs,
                f"Experiment: epsilon {eps_idx}/{len(epsilons)}, trial {trial+1}/{n_trials}"
            )
            rng = np.random.default_rng(seed + trial)
            X = subsample(X_full, n, rng)
            C_true = true_covariance(X)

            t0 = time.perf_counter()
            lap_est = dp_laplace(X, epsilon, k, rng)
            time_totals["Laplace"] += time.perf_counter() - t0
            time_counts["Laplace"] += 1
            errs_a["Laplace"].append(frobenius_error(C_true, lap_est))

            t0 = time.perf_counter()
            kt_est = dp_kt(X, epsilon, 1e-5, k, rng)
            time_totals["KT"] += time.perf_counter() - t0
            time_counts["KT"] += 1
            errs_a["KT"].append(frobenius_error(C_true, kt_est))

            t0 = time.perf_counter()
            alg1_est = dp_algorithm1_rank_k(X, epsilon, 1e-5, k, rng=rng)
            time_totals["Alg1"] += time.perf_counter() - t0
            time_counts["Alg1"] += 1
            errs_a["Alg1"].append(frobenius_error(C_true, alg1_est))

            for delta in gaussian_deltas:
                label = f"G-{int(np.log10(delta))}"
                t0 = time.perf_counter()
                g_est = dp_gaussian(X, epsilon, delta, k, rng)
                time_totals["Gaussian"] += time.perf_counter() - t0
                time_counts["Gaussian"] += 1
                errs_b[label].append(frobenius_error(C_true, g_est))
            errs_b["Alg1"].append(
                frobenius_error(C_true, dp_algorithm1_rank_k(X, epsilon, 1e-5, k, rng=rng))
            )
            completed_runs += 1

        for alg in results_a:
            results_a[alg].append(np.mean(errs_a[alg]))
        for alg in results_b:
            results_b[alg].append(np.mean(errs_b[alg]))
        _show_progress(completed_runs, total_runs,
                       f"Experiment: epsilon {eps_idx}/{len(epsilons)} complete")
        print(f"  eps={epsilon}: done")

    timing = {alg: time_totals[alg] for alg in time_totals}
    return results_a, results_b, timing

## 10. Plotting Helpers

In [ ]:
COLORS  = {"Laplace": "tab:blue", "Gaussian": "tab:orange", "KT": "tab:green",
           "Alg1": "tab:red", "G--16": "#c17d11", "G--10": "#e69f00", "G--3": "#f4c542"}
LABELS  = {"Laplace": "Laplace", "Gaussian": "Gaussian", "KT": "KT",
           "Alg1": "Algorithm 1 (Ours)", "G--16": "G-16", "G--10": "G-10", "G--3": "G-3"}
MARKERS = {"Laplace": "o", "Gaussian": "s", "KT": "^", "Alg1": "D",
           "G--16": "s", "G--10": "s", "G--3": "s"}


def plot_results(xs, results, xlabel, title, ax):
    for alg, vals in results.items():
        ax.plot(xs, vals,
                label=LABELS.get(alg, alg),
                color=COLORS.get(alg, None),
                marker=MARKERS.get(alg, "o"),
                linewidth=1.8, markersize=5)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel("Normalized Frobenius Error", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

## 11. Run Experiments

In [ ]:
DELTA          = 1e-5
N_TRIALS       = 50
ADULT_TRIALS   = 50
EPSILONS       = [0.01, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0]
GAUSSIAN_DELTAS = [1e-16, 1e-10, 1e-3]

datasets = [
    ("Wine",  X_wine),
    # ("Airfoil", X_airfoil),
    ("Adult", X_adult),
]

# Row 0: L/KT/AD panels; Row 1: G-δ/AD panels; one column per dataset
fig, axes = plt.subplots(2, len(datasets), figsize=(5 * len(datasets), 9))
if len(datasets) == 1:
    axes = np.array([[axes[0]], [axes[1]]], dtype=object)

json_output = {"epsilons": EPSILONS, "gaussian_deltas": GAUSSIAN_DELTAS, "datasets": {}}

for col, (name, X_full) in enumerate(datasets):
    k        = X_full.shape[0]
    n_use    = X_full.shape[1]
    n_trials = ADULT_TRIALS if name == "Adult" else N_TRIALS
    print(f"\n[{name}] d={X_full.shape[0]}, n={n_use}, k={k}...")

    res_left, res_right, timing = run_experiment_vary_eps_gaussian_grid(
        X_full, EPSILONS, n_use, GAUSSIAN_DELTAS, k, n_trials=n_trials
    )

    plot_results(EPSILONS, res_left,  "epsilon", f"{name}: L / KT / AD",     axes[0, col])
    plot_results(EPSILONS, res_right, "epsilon", f"{name}: G-delta / AD",    axes[1, col])

    print(f"\n  [{name}] Total time (seconds):")
    for alg, t in timing.items():
        print(f"    {alg:12s}: {t:.2f}s")

    json_output["datasets"][name] = {
        "shape":   list(X_full.shape),
        "k":       k,
        "n_trials": n_trials,
        "results_left":  {alg: [float(v) for v in vals] for alg, vals in res_left.items()},
        "results_right": {alg: [float(v) for v in vals] for alg, vals in res_right.items()},
        "timing_seconds_total": {alg: float(t) for alg, t in timing.items()},
    }

plt.suptitle("DP Covariance Estimation — Figure 1 Style Reproduction", fontsize=13)
plt.tight_layout()
plt.savefig("results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved results.png")

## 12. Save Results to JSON

In [ ]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(json_output, f, indent=2)
print("Saved results.json")